# ACWM｜Trajectory 实验入口
从 01 到 06 依次运行。新实验只改 02 的参数和实验名；续跑或评估旧实验保留原参数。正式实验使用 LeWorld 官方全量 Push-T。01 会将 Drive 压缩包解压到本次运行时（43.1 GiB），不重新下载；同一运行时重复运行会复用。旧实验保持原路径。先确认第 1 轮权重真的出现在 Drive 网页，再续训。

In [ ]:
# @title 01｜初始化运行时并解压官方数据
import importlib, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
repo = Path('/content/ACWM')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/StrawberryChen/ACWM.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[colab-trajectory]'], check=True)
from experiments.continuous_dynamics.leworld_data import prepare_colab_leworld
prepare_colab_leworld()
from experiments.continuous_dynamics import workflow
importlib.reload(workflow)
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print('如需 W&B 在线记录，请在 Colab Secrets 添加 WANDB_API_KEY 并允许此 Notebook 访问，或使用已登录会话。')

In [ ]:
# @title 02｜实验参数（新实验换 run_name）
variant = 'velocity_obs5_bn' # @param ['velocity_obs5_bn', 'velocity_obs5_cross_attn', 'velocity_sigreg', 'action_per_step', 'state_action_per_step', 'action_global', 'state_action_global']
run_name = 'velocity_obs5_bn_01' # @param {type:'string'}
epochs = 20 # @param {type:'integer'}
batch_size = 32 # @param {type:'integer'}
learning_rate = 0.0003 # @param {type:'number'}
sigreg_weight = 0.09 # @param {type:'number'}
velocity_euler_steps = 2 # @param {type:'integer'}
legacy_progress_weight = 0.1 # @param {type:'number'}
val_every = 5 # @param {type:'integer'}
wandb_enabled = True # @param {type:'boolean'}
planning_episodes = 10 # @param {type:'integer'}
planning_max_steps = 100 # @param {type:'integer'}
from experiments.continuous_dynamics.workflow import configure_colab
overrides = {
    'data': {'batch_size': batch_size},
    'training': {'epochs': epochs, 'learning_rate': learning_rate},
    'validation': {'interval': val_every},
    'wandb': {'enabled': wandb_enabled},
}
if variant.startswith('velocity_'):
    overrides['loss'] = {'sigreg_weight': sigreg_weight}
    overrides['model'] = {'integration_steps': velocity_euler_steps}
else:
    overrides['loss'] = {'weight': legacy_progress_weight}
flow = configure_colab(variant, run_name, overrides=overrides,
    planning={'episodes': planning_episodes, 'max_steps': planning_max_steps})
flow.preflight()

In [ ]:
# @title 03｜训练第 1 轮 / 检查已有权重
flow.first_epoch()
print('到 Drive 网页确认上面打印的目录里有 checkpoints/latest.pt，再运行 04。')

In [ ]:
# @title 04｜确认云端权重后，续训到目标轮数
drive_confirmed = False # @param {type:'boolean'}
flow.train_remaining(drive_confirmed=drive_confirmed)

In [ ]:
# @title 05｜Probe（读取本次最终权重）
flow.probe()

In [ ]:
# @title 06｜Planning（真实 Push-T；结果和视频写 Drive）
# Obs5 Euler 对照：分别运行 flow.plan(euler_steps=1) 和 flow.plan(euler_steps=2)。
flow.plan()